### The simulations published used 300-400 CPUs for a couple of days using Julias pmap feature with workers called via SSH.
### The data frame that's being used is JLD2, which actually uses HDF5, allowing to load data also in python efficiently.

In [1]:
using Distributed
using Parallelism
using ProgressMeter

TARGET_WORKERS = 400
#write here your code to add workers

400

In [2]:
#needed for robust_pmap to work together with ProgressMeter, this is from the package Parallelism.jl
ProgressMeter.ncalls(::typeof(robust_pmap), f::Function, args...) =
    ProgressMeter.ncalls(pmap, f, args...)


# Start main simulation

In [3]:
@everywhere begin

using Combinatorics
using ProgressMeter
using Random
using StatsBase

end

#only needed for saving the data in the master process
using JLD2
using Dates

# Code to load, save and check data

In [4]:
# Helper to ensure consistent naming conventions
get_group_name(L::Int, A::Int) = "L$(L)_A$(A)"
get_group_vals(s) = match(r"L(\d+)_A(\d+)", s)

"""
    save_LA_dict(filepath, L, A, data_dict)

Stores the values from `data_dict` into the file under the specific L and A group.
If a key already exists, the new data is appended to the existing array.
"""
function save_LA_dict(filepath::String, L::Int, A::Int, data_dict::Dict)
    group_name = get_group_name(L, A)
    
    # "a+" creates the file if it doesn't exist, and appends if it does
    jldopen(filepath, "a+") do file
        # Initialize group for this L and A if not present
        if !haskey(file, group_name)
            JLD2.Group(file, group_name)
        end
        
        grp = file[group_name]
        
        for (k, v) in data_dict
            key_str = String(k)
            # Append if key already exists
            if haskey(grp, key_str)
                existing_data = grp[key_str]
                new_data = vcat(existing_data, v)
                delete!(grp, key_str)
                grp[key_str] = new_data
            else
                grp[key_str] = v
            end
        end
    end
    #println("Saved $(length(data_dict)) keys to $group_name in $filepath")
end

"""
    get_sample_count(filepath, L, A, target_key)

Returns the length of the array connected to `target_key` to check how many samples exist. 
Returns 0 if the file, group, or key does not exist.
"""
function get_sample_count(filepath::String, L::Int, A::Int, target_key::String)::Int
    group_name = get_group_name(L, A)
    
    if !isfile(filepath)
        return 0
    end
    
    jldopen(filepath, "r") do file
        if haskey(file, group_name) && haskey(file[group_name], target_key)
            return length(file[group_name][target_key])
        else
            return 0
        end
    end
end

"""
    load_LA_keys(filepath, L, A, keys_to_load)

Loads a specified list of keys into a Dictionary. 
If `keys_to_load` is empty, it loads all available keys for that L and A.
"""
function load_LA_keys(filepath::String, L::Int, A::Int, keys_to_load::Vector{String}=String[])
    group_name = get_group_name(L, A)
    result = Dict{String, Any}()
    
    if !isfile(filepath)
        @warn "File $filepath does not exist."
        return result
    end
    
    jldopen(filepath, "r") do file
        if !haskey(file, group_name)
            @warn "No data found for $group_name"
            return result
        end
        
        grp = file[group_name]
        target_keys = isempty(keys_to_load) ? keys(grp) : keys_to_load
        
        for k in target_keys
            if haskey(grp, k)
                result[k] = grp[k]
            else
                @warn "Key '$k' is missing in $group_name"
            end
        end
    end
    
    return result
end


load_LA_keys

# Standart dfs, passing the whole fitness landscape as a argument.

In [5]:
@everywhere begin

"""
    AdB_size_dfs(peak::Int, ω::Vector{<:Real}; L::Int, A::Int)

Return the size of the adaptive basin of a peak genotype in a
multi-allelic Hamming graph (L loci, A alleles each).

Each genotype `i ∈ 0:(A^L-1)` is represented by its integer index.
A genotype belongs to the basin if it can reach `peak` through some
strictly increasing fitness path.

Depth-first search (DFS) is used for low memory; only a visited bitvector
and a small stack are kept in memory.
"""
function AdB_size_dfs(v_start, ω, L, A)
    N = length(ω)
    @assert N == A^L "Length of ω must equal A^L"

    powers = [A^k for k in 0:(L-1)]
    visited = falses(N)
    stack = Vector{Int}(undef, 0)
    push!(stack, v_start)
    count = 0

    while !isempty(stack)
        i = pop!(stack)
        if visited[i + 1]
            continue
        end
        visited[i + 1] = true
        count += 1

        fi = ω[i + 1]

        # Loop over loci
        @inbounds for k in 1:L
            base = powers[k]
            allele = (i ÷ base) % A
            # Try all alternative alleles at locus k
            for a in 0:(A - 1)
                a == allele && continue
                j = i - allele * base + a * base
                if !visited[j + 1] && ω[j + 1] < fi
                    push!(stack, j)
                end
            end
        end
    end
    
    return count
end

#checks if a vertex is a peak
function is_peak(ω, v, L, A)::Bool
    f_g = ω[v+1]
    is_peak::Bool = true

    @inbounds for l in 0:L-1
        base = (v ÷ A^l) % A
        for a in 0:A-1
            a == base && continue
            neighbor = v + (a - base) * A^l
            if ω[neighbor+1] > f_g
                is_peak = false
                break
            end
        end
        is_peak || break
    end

    return is_peak
end

#main loop
function calc_AdB_array_size(seed::UInt, a_v, L, A)
    len = length(a_v)
    ω = sample_fitness_array(seed, L, A)

    len = length(a_v)
    a_AdB_size = Array{Int64}(undef, len)
    a_is_peak = Array{Bool}(undef, len)
    a_fitness = Array{Float64}(undef, len)

    #iterate trough all vertices in question
    for (i, v) in enumerate(a_v)
        a_is_peak[i] = is_peak(ω, v, L, A)
        a_AdB_size[i] = AdB_size_dfs(v, ω, L, A)
        a_fitness[i] = ω[v+1]
    end

    return a_AdB_size, a_fitness, a_is_peak
end

#function which samples the fitness values from a given seed, in order to not write it to many times
function sample_fitness_array(seed::UInt, L, A)
    rng = Xoshiro(seed)
    
    return rand(rng, A^L)
end

end


In [6]:
#We want a specific number of peaks which are sampled without a bias
#Use rejection sampling to get a random sample of peaks if the landscape is quite large

#function which gives back an array of all v values of peaks in the landscape between v_min and v_max (default is the whole landscape)
function get_peaks(seed::UInt, L, A, v_min = -1, v_max = -1)
    rng = Xoshiro(seed)
    ω = rand(rng, A^L)

    v_min == -1 && (v_min = 0)
    v_max == -1 && (v_max = A^L - 1)

    peaks = Int[]
    for v in v_min:v_max
        if is_peak(ω, v, L, A)
            push!(peaks, v)
        end
    end

    return peaks
end

#tries to find a specific number of peaks by sampling random genotypes and checking if they are peaks
#until the target number is reached or a safe upper bound of attempts is reached
#returns a unique number of peaks
#Only use it for rough landscapes, in others it takes a long time to find peaks
function sample_random_peaks(seed::UInt, num_peaks_to_find, L, A;
                            upper_bound_peaks=0.5, upper_bound_attempts=0.2)
    max_state = A^L
    
    # Calculate expected peaks. We use a safe upper bound of upper_bound of the expected total number of peaks.
    n = L*(A - 1)
    expected_peaks = max_state / (n+1)
    safe_target = min(num_peaks_to_find, floor(Int, expected_peaks * upper_bound_peaks))
    safe_target = max(safe_target, 1) #if safe_target is zero, set it to 1 to ensure we try to find at least one peak
    max_attempts = Int(ceil(max_state * upper_bound_attempts))

    a_peaks = Int64[]
    guess_rng = Xoshiro(seed + 0x1234567) 
    
    attempts = 0
    peaks_found = 0

    ω = sample_fitness_array(seed, L, A)

    while length(a_peaks) < safe_target && attempts < max_attempts
        v_guess = rand(guess_rng, 0:(max_state - 1))
        attempts += 1
        
        if is_peak(ω, v_guess, L, A)
            push!(a_peaks, v_guess)
            peaks_found += 1
        end
    end
    
    return unique!(a_peaks)
end

function get_peak_sample(seed::UInt, L, A, n_calc)

    #if a landscape is too small, simply get all peaks, otherwise use the sampling function

    a_v = Int64[]

    if A^L <= 10000
        a_v_peaks = get_peaks(seed, L, A)
        a_v = sample(a_v_peaks, min(n_calc, length(a_v_peaks)), replace=false)
    else
        #try this here out until there is atleast a single peak found, otherwise there is no use
        while length(a_v) == 0
            a_v = sample_random_peaks(seed, n_calc, L, A)
            if length(a_v) == 0
                println("No peaks found for L: $L | A: $A | seed: $seed, resampling...")
            end
        end
    end

    return a_v
end

#=
#test the above function
seed = rand(UInt)
L = 5
A = 4
n_calc = 10
a_v = get_peak_sample(seed, L, A, n_calc)
=#


get_peak_sample (generic function with 1 method)

In [7]:
#calculate the maximal values of L and A given a memory limit of 2GB (using only the fitness array)

nGiB_max = 0.0001 #2
a_LA_combinations = Tuple{Int, Int}[]

a_L_givenA = Dict{Int, Vector{Int}}([
    2 => collect(5:24),
    3 => collect(5:14),
    4 => collect(4:10),
    10 => collect(2:6), #see line below
    20 => collect(2:5)]) #the maximal number for both of them was chosen such that the computation time does not completely blows up, as tested on approx. 400 threads

for A in [2, 3, 4, 10, 20]
    for L in a_L_givenA[A]
        nGiB = (BigInt(A)^L * 8) / 1024^3
        if nGiB <= nGiB_max
            push!(a_LA_combinations, (L, A))
        end
    end
end

#sort combinations by #number of genotypes + #number of edges
sort!(a_LA_combinations, by = x -> x[1]^x[2]) #* (1 + x[1] * (x[2] - 1) / 2))
nothing
println(a_LA_combinations)
println("Total combinations: ", length(a_LA_combinations))

[(5, 2), (6, 2), (7, 2), (8, 2), (9, 2), (10, 2), (11, 2), (5, 3), (12, 2), (13, 2), (6, 3), (4, 4), (7, 3), (8, 3), (5, 4), (2, 10), (6, 4), (3, 10), (4, 10), (2, 20), (3, 20)]
Total combinations: 21


In [8]:
#This here is a loop to stop the simulation at a spcific time, to free up the resources
#=
cutoff_time = DateTime(year, month, day, h, m, s)

@async begin
    while now() < cutoff_time
        sleep(60)
    end

    #Loop to kill the workers and master after the cutoff time is reached
    @sync begin
        for w in workers()
            @async try
                # rmprocs sends a direct SIGTERM to the worker OS process
                rmprocs(w; waitfor=5.0) 
            catch
                println("Worker $w failed to shut down gracefully.")
            end
        end
    end
    
    println("All workers cleared. Shutting down master.")
    
    # Step 3: Now it is safe to kill the master
    exit(0)
end
=#

In [ ]:
out_data = joinpath(@__DIR__, "data")
mkpath(out_data)
file_name = "/AdB_data_thp_cluster.jld2"
#create file if it does not exist
isfile(out_data * file_name) || jldopen(out_data * file_name, "w") do file end

nSamples = 10^7 #number used for the full range variables
nMaxSamplesPerLandscape = 10^4
nBatchSizeMax = 10^2 #maximum number of genotypes to check per batch

#we want to scale the number of landscapes such that we check 10^6 genotypes in total
total_len = length(a_LA_combinations)

for (i_run, (L, A)) in enumerate(a_LA_combinations)
    #check if the (L, A) combination exists and if so skip it
    f_skip = false


    jldopen(out_data * file_name, "r") do file
        group_name = get_group_name(L, A)
        if haskey(file, group_name)
            f_skip = true
        end
    end
    if f_skip
        continue
    end

    n_calc = minimum([nMaxSamplesPerLandscape, ceil(Int, 0.01 * A^L)])
    n_neigh = L * (A - 1)
    #n_calc = minimum([nMaxSamplesPerLandscape, ceil(Int, 0.25*A^L/(n_neigh+1))])

    #array that contains the seed and a_v
    a_batches = Vector{Tuple{UInt, Vector{Int}}}(undef, 0)

    nLandscapes = 0 #ceil(Int, nSamples / n_calc)
    #perform this loop until we have in total nSamples genotypes to check
    #@showprogress for i in 1:nLandscapes
    nSamples_current = 0
    while nSamples_current < nSamples
        #create individual batches. Each batch contains the seed for this landscape and the v values of the genotypes to check
        seed = rand(UInt)
        nLandscapes += 1

        #we only check a fraction of the landscape.
        a_v = unique!(sample(0:(A^L - 1), n_calc, replace=false)) #checking random genotypes
        #a_v = get_peak_sample(seed, L, A, n_calc)

        nSamples_current += length(a_v)

        #split a_v into batches
        a_v_batches = [a_v[i:min(i+nBatchSizeMax-1, end)] for i in 1:nBatchSizeMax:length(a_v)]

        #combine the seed and the batches into a single array
        for k in a_v_batches
            push!(a_batches, (seed, k))
        end
    end
    nBatches = length(a_batches)

    #println("L: $L|A: $A| W: $(nworkers()) | B:$nBatches | L: $nLandscapes")
    #continue

    λ_f = x -> calc_AdB_array_size(x[1], x[2], L, A) #anonomous function to call the calculation for a batch
    res = @showprogress dt=5.0 "T:$i_run/$total_len|L:$L|A:$A|W:$(nworkers())|B:$nBatches|L:$nLandscapes" robust_pmap(λ_f, a_batches; num_retries=100)

    #save data
    d_save = Dict{String, Any}()
    d_save["a_AdB_size"] = [r[1] for r in res]
    d_save["a_fitness"] = [r[2] for r in res]
    d_save["a_is_peak"] = [r[3] for r in res]
    d_save["a_v_batches"] = [b[2] for b in a_batches]
    d_save["a_seeds"] = [b[1] for b in a_batches]
    save_LA_dict(out_data * file_name, L, A, d_save)
end


# Plot to check some of the data

In [18]:
#=
using Plots
using Roots

# load the data and plot the results
A = 2

plt = Plots.plot()

nBins = 100

#load all corresponding L values

for (L, A_tmp) in a_LA_combinations
    A_tmp == A || continue
    ret = load_LA_keys(out_data * file_name, L, A, ["a_AdB_size", "a_fitness"])
    ret == 0 && break
    #println("L: $L | A: $A | ", length(ret["a_AdB_size"]), " samples")

    #calculate the mean AdB size for each fitness bin
    fitness_bins = range(0, 1, length=nBins+1)
    a_bins = [Int64[] for _ in 1:nBins]

    #group into the bins and calculate the mean AdB size for each bin
    for (a_AdB_size, a_AdB_fitness) in zip(ret["a_AdB_size"], ret["a_fitness"])
        for (AdB_size, AdB_fitness) in zip(a_AdB_size, a_AdB_fitness)
            #sort into the corresponding bin
                bin_index = findfirst(x -> x > AdB_fitness, fitness_bins) - 1
                #bin_index = max(bin_index, 1) # Ensure it doesn't go below 1
                push!(a_bins[bin_index], AdB_size)
        end
    end

    mean_AdB_size = [mean(bin) / A^L for bin in a_bins]
    Plots.plot!(fitness_bins[1:end-1], mean_AdB_size, label="L=$L")
end

Plots.xlabel!("Fitness")
Plots.ylabel!("Mean Adaptive Basin Size")
Plots.title!("Mean Adaptive Basin Size vs Fitness for A=$A")

δ_star = (A-1)/A
Γ_full(β, δ, A) = -log(A) - β + δ * log(exp(A*β) - 1) + (1-δ) * log(exp(A * β) + A - 1)
β_star = find_zero(β -> Γ_full(β, δ_star, A), (0, 1))


#add linear line given by ω-β_star
ω_range = range(β_star, 1, length=1000)
Plots.plot!(ω_range,  ω_range .- β_star, label="ω - β*")
=#